# Explicit dyes: labelling, rotamer libraries and FRET

`IMP.bff` models a fluorescent label in two ways. The **accessible volume** (AV,
see *Accessible volumes*) treats the dye as a point on a flexible linker and
enumerates where it can go. The **explicit dye** route (the `cgprobe` layer)
places an atomistic dye + linker on a residue, either from a **rotamer library**
(FRETpredict's libraries ship as module data) or by **sampling the linker
degrees of freedom**, and scores each conformer against the protein. This page
walks the explicit route with the flat `IMP.bff.*` API; everything shown here is
reachable as `IMP.bff.<Name>` (the sub-packages are just where the code lives).

Physics pins behind this page (PRD-107): rotamer FRET reproduces FRETpredict on
its Hsp90 and pp11 references (`test/cgprobe/rotamer/test_fretpredict_pins.py`),
and the invariants of κ², R0, the FRET regimes and the kinetic master equation
are asserted in `test/cgprobe/test_physics_invariants.py`.


## 1. Label a structure with an explicit dye

`attach_probes` moves a dye hierarchy (read from MOL2 or PDB) into the backbone
frame of a residue (origin CA, x along CA→N, y in the N–CA–C plane) and, on
request, strips the residue's side chain first. Note the keep-set: cgprobe keeps
`N CA C O OXT` and **strips CB**, because the explicit linker is built off CA
and replaces the whole side chain; the AV convention keeps CB. Both go through
one strip engine (`IMP.bff.strip_hierarchy`, PRD-106).


In [1]:
import json
import pathlib
import IMP, IMP.atom, IMP.core
import IMP.bff
from IMP.bff import get_structure_dir   # bundled inputs (data/cgprobe)

m = IMP.Model()
protein = IMP.atom.read_pdb(str(get_structure_dir("1DG3.pdb")), m, IMP.atom.NonWaterPDBSelector())
dye = IMP.atom.read_mol2(str(get_structure_dir("alexa488_r48.mol2")), m)

site = IMP.bff.resolve_probe_site(protein, "A", 481)          # {'CA':..., 'N':..., 'C':...}
attachment = IMP.bff.ProbeAttachment(dye, "A", 481)
attached = IMP.bff.attach_probes(protein, [attachment],
                                 strip_site_sidechain=True)
print("dye atoms:", len(IMP.atom.get_by_type(dye, IMP.atom.ATOM_TYPE)),
      "site keep-set:", list(IMP.bff.site_keep_atom_names()))


dye atoms: 83 site keep-set: ('N', 'CA', 'C', 'O', 'OXT')


## 2. Rotamer libraries

The FRETpredict rotamer libraries are IMP.bff module data
(`data/rotamer_library`). A library name selects the dye, the linker and the
clustering cutoff; the cutoff in the name is honoured (`cutoff10` = 711
rotamers for Alexa488 C1R, `cutoff30` = 33 — the default).

The libraries ship as one **`.drot.pto`** container per family (PRD-118):
a PTO/EBML document holding, per library, the template, the Z-matrix and, per
conformer, its own base coordinates, bond lengths, bond angles and dihedrals. Two things follow. It is *self-contained*
— atom names, residue names and elements ride inside it, so unlike the
BinaryCIF frame store it needs no `.pdb` beside it — and a conformer *is* a
dihedral vector, which is what continuous side-chain sampling needs. It is
also lossless: rebuilding a conformer reproduces the source to ~1e-6 Å, which
is what keeps the FRETpredict parity pins inside their 1e-5 tolerance, at
~0.65× the size of the BinaryCIF of the same ensemble (which still ships, and
still reads).

Building a library from a trajectory is one program — `.bcif`, `.dcd` and
(through mdtraj) `.xtc` all read:

```
imp_bff_traj2drot lib.bcif lib.drot --top lib.pdb --weights lib_weights.txt
imp_bff_traj2drot raw.dcd  lib.drot --top lib.pdb --cluster 1.0
```

`--cluster A` takes PRD-118's raw-MD path: the frames are leader-clustered by
`cluster_frames_leader` — the algorithm FRETpredict's own libraries were built
with — the leaders become the rotamers and the cluster populations the
weights. A path is a library name everywhere a name is taken, so a library you
just built drops straight into `ProbeRotamerEnsemble.from_site` and `FRETRotamer`;
`examples/structure/drot_rotamer_library.py` runs that path end to end.

In [2]:
lib = IMP.bff.load_probe_rotamer_library("AlexaFluor 488 C1R cutoff30")
meta = json.loads(lib.metadata)
print(lib.coords.shape, "rotamers x atoms x 3;", "weights sum", lib.weights.sum())
print("chromophore centre selector:", meta["r"], " dipole:", meta["mu"])
print(pathlib.Path(IMP.bff.resolve_probe_rotamer_library_path("AlexaFluor 488 C1R cutoff10")).name)

(33, 83, 3) rotamers x atoms x 3; weights sum 1.0
chromophore centre selector: ['C7 and resname A48']  dipole: ['C2', 'C13 and resname A48']
dyes.drot.pto::A48_C1R_cutoff10


## 3. Förster radius and κ²

R0 comes from the bundled donor emission / acceptor excitation spectra
(`IMP.bff.forster_radius_from_spectra`) and is in **Ångström**, like every
other length in this package -- the spectra it integrates are in nanometres
and the conversion happens once, inside. κ² comes from transition-dipole
vectors (`IMP.bff.kappa2_dipole_matrix`, the full donor-by-acceptor matrix).


In [3]:
import numpy as np
r0_iso = IMP.bff.forster_radius_from_spectra("AlexaFluor 488", "AlexaFluor 594", IMP.bff.kappa2_isotropic())
print("R0(A488/A594, kappa2=2/3) =", round(r0_iso, 3), "A")
mu_d = np.array([[0.0, 0.0, 1.0]]); mu_a = np.array([[0.0, 0.0, 1.0]]); r = np.array([[[0.0, 0.0, 50.0]]])
print("collinear kappa2 =", IMP.bff.kappa2_dipole_matrix(mu_d.ravel(), mu_a.ravel(), r.ravel())[0])


R0(A488/A594, kappa2=2/3) = 56.878 A
collinear kappa2 = 4.0


## 4. Rotamer FRET (FRETpredict-compatible)

`FRETRotamer` places two libraries on two sites of a structure (PDB, multi-MODEL
PDB, or RMF trajectory), screens every rotamer against the protein
(Lennard-Jones, optional Debye–Hückel), and reports per frame the static,
dynamic-1 and dynamic-2 efficiencies, ⟨κ²⟩ and the partition functions.


In [4]:
from IMP.bff import get_structure_dir
fret = IMP.bff.FRETRotamer(
    str(get_structure_dir("148L.pdb")), [22, 137], chains=["E", "E"],   # T4 lysozyme, chain E
    donor="AlexaFluor 488", acceptor="AlexaFluor 594",
    libname_1="AlexaFluor 488 C1R cutoff30", libname_2="AlexaFluor 594 C1R cutoff30",
    temperature=298, electrostatic=True, output_prefix="t4l_rotamer")
fret.trajectory_analysis()
print("E_static", fret.estatic_values, "E_dyn1", fret.edynamic1_values, "<kappa2>", fret.k2_values, "Z", fret.z_values)


E_static [0.9961626] E_dyn1 [0.99853722] <kappa2> [0.63724536] Z [[4.04320651 0.5739155 ]]


## 5. FRET regimes and the kinetic master equation

Given a donor×acceptor grid of distances and κ² with weights,
`fret_efficiency_regimes` returns the static, dynamic and dynamic+ averages;
`fret_efficiency_exact_kinetic` solves the master equation for a transition
matrix (slow exchange → static average, fast exchange → dynamic average).


In [5]:
d = np.array([[45.0, 60.0], [52.0, 48.0]]); k2 = np.array([[0.5, 1.2], [0.9, 2.0]])
wd = np.array([0.7, 0.3]); wa = np.array([0.4, 0.6])
print(IMP.bff.fret_efficiency_regimes(d, k2, wd, wa, forster_radius=52.0))
P = IMP.bff.transition_matrix_from_counts([8, 2, 3, 7], 2).reshape(2, 2)
rates = ((52.0 / d[:, 0]) ** 6 * 1.5 * k2[:, 0] / 4.0)      # donor states vs one acceptor, 1/ns
print("E (kinetic, donor exchange):", IMP.bff.fret_efficiency_exact_kinetic(P, rates, tau0=4.0, dt=0.1))


FRETRegimes(static=0.579387, dynamic=0.603931)
E (kinetic, donor exchange): 0.6167689346520641


## 6. Sampling the linker yourself

`LinkerSampler` / `generate_linker_rotamers` Metropolis-sample a dye's linker
torsions and bond angles (bonded 1-2/1-3/1-4 pairs excluded from the LJ score),
cluster the frames and Boltzmann-weight the clusters; `run_torsion_rrt` and
`run_rigid_body_rrt` grow collision-free trees. Real stochastic dynamics of an
attached dye -- Langevin-thermostat MD or Brownian dynamics on the dye force
field with the protein as soft-sphere obstacles -- is `AttachedProbeDynamics`
(`integrator="md"|"bd"`; thermodynamic pins in `test/cgprobe/test_langevin_sampler.py`).
The command line (`imp_bff --help`) drives the same code;
`examples/structure/hgbp1_explicit_dye_fret.py` strings ensembles, FRET, an
fps.json with `R1` positions and a Langevin run together.


In [6]:
lib_gen = IMP.bff.generate_linker_rotamers(str(get_structure_dir("alexa488_r48.mol2")),
                                          n_steps=200, write_every=10, cluster_threshold=1.0, seed=42)
# `generate_linker_rotamers` answers an `IMP.bff.ProbeRotamerLibrary` -- the one
# value every rotamer reader in this module returns -- not a dict.
print(lib_gen.n_rotamers, "clusters;",
      lib_gen.n_atoms, "atoms; transitions", sum(lib_gen.transitions))

# a short Langevin run of the attached dye (see section 1 for `protein`, `dye`)
sampler = IMP.bff.AttachedProbeDynamics(protein, dye, str(get_structure_dir("alexa488_r48.mol2")), "A", 481,
                                     integrator="md", temperature=300.0, seed=1)
sampler.minimize(100)
traj = sampler.run(1000, write_every=100)
print(traj.n_frames, "frames; <T_kin> =", round(sum(sampler.kinetic_temperature(k) for k in traj.kinetic_energy) / traj.n_frames), "K")


20 clusters; transitions 19


10 frames; <T_kin> = 300 K
